[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/02-python-for-data-science/pyds-eda-multivariate.ipynb)

# EDA Part 2: Bivariate & Multivariate Analysis

*AIBits Academy · Machine Learning End To End · Python For Data Science*

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

This lesson continues from *EDA Part 1*. That page created `Cleaned_Employee_Details.csv`; the cell below rebuilds it (same code and seed) so this notebook runs on its own.

In [ ]:
import numpy as np, pandas as pd, random

np.random.seed(42); random.seed(42)
n = 50
departments = ['Sales', 'Marketing', 'HR', 'Finance', 'IT']
qualifications = ['High School', 'Bachelor', 'Master', 'PhD']
marital = ['Single', 'Married', 'Divorced', 'Widowed']

def normal_with_outliers(size, mean, std, low, high):
    data = np.random.normal(mean, std, size)
    med = np.median(data)
    data[:low] = med - 3 * std
    data[low:low + high] = med + 3 * std
    return data

df = pd.DataFrame({
    'Employee_ID': range(1000, 1000 + n),
    'Salary': normal_with_outliers(n, 50000, 10000, 2, 3).astype(int),
    'Age': normal_with_outliers(n, 35, 5, 2, 3).astype(int),
    'Performance': normal_with_outliers(n, 7, 1.5, 2, 3).astype(int),
    'Qualification': random.choices(qualifications, k=n),
    'Marital_Status': random.choices(marital, k=n),
    'Department': random.choices(departments, k=n),
})
df = pd.concat([df, df.sample(n=5, random_state=42)], ignore_index=True)
for _ in range(15):
    idx = np.random.randint(0, df.shape[0])
    col = random.choice(df.columns.tolist())
    df.loc[idx, col] = np.nan

df = df.drop_duplicates()
for c in df.select_dtypes(include=['int64', 'float64']).columns:
    df[c] = df[c].fillna(df[c].median())
for c in df.select_dtypes(include=['object']).columns:
    df[c] = df[c].fillna(df[c].mode()[0])

def cap_outliers(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return np.clip(s, q1 - 1.5 * iqr, q3 + 1.5 * iqr)

for c in ['Salary', 'Age']:
    df[c] = cap_outliers(df[c])
df.to_csv('Cleaned_Employee_Details.csv', index=False)
print("rebuilt Cleaned_Employee_Details.csv", df.shape)

Once each variable is understood on its own, the interesting questions are about *relationships*: does salary rise with age? Which departments pay most? Bivariate and multivariate analysis answer these with scatter plots, correlations, grouped comparisons, and heatmaps.

We pick up the cleaned employee table saved on the previous page.

In [ ]:
import pandas as pd

df = pd.read_csv('Cleaned_Employee_Details.csv')
print(df.shape)

## Two Numeric Variables: Scatter & Correlation

A **scatter plot** shows how two numeric variables move together; the **Pearson correlation** quantifies it on a −1…+1 scale.

In [ ]:
print("correlation (Age vs Salary):", round(df['Age'].corr(df['Salary']), 4))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.scatterplot(x='Age', y='Salary', data=df)
plt.title('Age vs Salary')
plt.tight_layout()
plt.show()

A moderate positive correlation (r ≈ 0.48): older employees tend to earn more, but with plenty of spread.

## Many Numeric Variables at Once: Correlation Heatmap

A **heatmap** of the correlation matrix shows every pairwise relationship in one view — the fastest way to spot which variables move together.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = df[['Salary', 'Age', 'Performance']].corr()
print(corr.round(2))

sns.heatmap(corr, annot=True, cmap='PuOr', center=0, fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

All three variables are positively related; Salary–Age is the strongest pair at 0.48.

## Numeric vs Categorical: Grouped Box Plots

To compare a numeric variable *across* categories, a grouped box plot places one box per category side by side — instantly revealing which department pays most and how spread its salaries are.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(x='Department', y='Salary', data=df, hue='Department',
            palette='Purples', legend=False)
plt.title('Salary Distribution by Department')
plt.tight_layout()
plt.show()

Each box summarises one department's salary spread, making medians and ranges directly comparable.

## Two Categorical Variables: Crosstab & Grouped Bars

`pd.crosstab` counts the co-occurrence of two categorical columns; a grouped bar chart turns that table into a visual.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ct = pd.crosstab(df['Department'], df['Qualification'])
print(ct)

ct.plot(kind='bar', colormap='viridis', width=0.8)
plt.title('Qualification Count within each Department')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Each department's qualification mix side by side — HR skews toward High School, Finance toward advanced degrees.

## Multivariate: The Pair Plot

A **pair plot** draws every numeric pair as a scatter and each variable's distribution on the diagonal — a one-shot overview of all relationships at once.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.pairplot(df[['Salary', 'Age', 'Performance']], corner=True)
plt.suptitle('Pairwise Relationships', y=1.02)
plt.show()

The lower-triangle scatter grid plus diagonal histograms summarise all three variables and their pairwise links together.

> **✅ What You Can Now Do**
>
> You can quantify and visualise relationships between variables — scatter plots and correlation for numeric pairs, heatmaps for many at once, grouped box plots for numeric-vs-categorical, crosstabs and grouped bars for categorical pairs, and pair plots for a full multivariate overview. This completes the Python-for-Data-Science EDA toolkit.

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Correlation by hand and by pandas

`hours` studied and `marks` scored are below. Store the Pearson correlation in `r` (rounded to 3 decimals) and `direction` = `"positive"` or `"negative"`.

In [ ]:
import pandas as pd
study = pd.DataFrame({"hours": [1, 2, 3, 4, 5, 6], "marks": [35, 42, 50, 58, 61, 72]})
r = direction = None   # TODO


In [ ]:
try:
    check("r is 0.994", r == 0.994)
    check("direction", direction == "positive")
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import pandas as pd
study = pd.DataFrame({"hours": [1, 2, 3, 4, 5, 6], "marks": [35, 42, 50, 58, 61, 72]})
r = round(study["hours"].corr(study["marks"]), 3)
direction = "positive" if r > 0 else "negative"

```

</details>

### Exercise 2 · Medium · Row-normalised crosstab

Build `mix`: for each `Region`, the **share** (0–1) of orders coming from each `Channel` — a row-normalised `pd.crosstab`.

In [ ]:
import pandas as pd
orders = pd.DataFrame({"Region": ["N", "N", "N", "S", "S", "S", "S"], "Channel": ["web", "web", "app", "web", "app", "app", "app"]})
mix = None   # TODO


In [ ]:
try:
    check("rows sum to 1", mix is not None and (mix.sum(axis=1).round(6) == 1).all())
    check("North web share", round(mix.loc["N", "web"], 3) == 0.667)
    check("South app share", mix.loc["S", "app"] == 0.75)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import pandas as pd
orders = pd.DataFrame({"Region": ["N", "N", "N", "S", "S", "S", "S"], "Channel": ["web", "web", "app", "web", "app", "app", "app"]})
mix = pd.crosstab(orders["Region"], orders["Channel"], normalize="index")

```

</details>

### Exercise 3 · Stretch · Find the strongest relationship

`data` has four numeric columns. Store in `best` the pair (as a sorted tuple of two column names) with the highest absolute correlation, ignoring the diagonal.

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(3)
x = rng.normal(size=300)
data = pd.DataFrame({"a": x, "b": rng.normal(size=300), "c": -2 * x + rng.normal(scale=0.3, size=300), "d": rng.normal(size=300)})
best = None   # TODO


In [ ]:
try:
    check("a and c are tied together", best == ("a", "c"))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np, pandas as pd
rng = np.random.default_rng(3)
x = rng.normal(size=300)
data = pd.DataFrame({"a": x, "b": rng.normal(size=300), "c": -2 * x + rng.normal(scale=0.3, size=300), "d": rng.normal(size=300)})
c = data.corr().abs()
c = c.where(~np.eye(len(c), dtype=bool))
best = tuple(sorted(c.stack().idxmax()))

```

A negative correlation of −0.99 is just as strong as +0.99 — hence `abs()`.

</details>

---
*Back to the course: **Machine Learning End To End → EDA Part 2: Bivariate & Multivariate Analysis**.*